# compare CN greeks with BS greeks

In [1]:
import numpy as np
import scipy.stats
import pandas as pd


K = 100
S0 = 100
T = 0.5
r = 0.02
sigma = 0.3
s_steps = 500
t_steps = 500


## bs greeks


In [2]:

def _d1(S, K, r, sigma, T):
    return (np.log(S) - np.log(K) + (r + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))

def _d2(S, K, r, sigma, T):
    return _d1(S, K, r, sigma, T) - sigma * np.sqrt(T)

def _bs_delta(S, K, r, sigma, T, type='call'):
    d1 = _d1(S, K, r, sigma, T)
    if type == 'call':
        A = 1
    else:
        A = -1

    return A * scipy.stats.norm.cdf(A * d1)

def _bs_gamma(S, K, r, sigma, T):
    d1 = _d1(S, K, r, sigma, T)
    return scipy.stats.norm.pdf(d1) / (S * sigma * np.sqrt(T))

def _bs_theta(S, K, r, sigma, T, type='call'):
    d1 = _d1(S, K, r, sigma, T)
    d2 = _d2(S, K, r, sigma, T)

    if type == "call":
        A = 1
    else:
        A = -1

    bs_theta_term_1 = - A * K * np.exp(-r*T) * r * scipy.stats.norm.cdf(A * d2)
    bs_theta_term_2 = - S * sigma * scipy.stats.norm.pdf(d1) / (2 * np.sqrt(T))

    return bs_theta_term_1 + bs_theta_term_2

def bs_greeks(S, K, r, sigma, T, type):
    delta = _bs_delta(S, K, r, sigma, T, type=type)
    gamma = _bs_gamma(S, K, r, sigma, T)
    theta = _bs_theta(S, K, r, sigma, T, type=type)
    return delta, gamma, theta

In [3]:
from src.options import EuropeanOption

call_option = EuropeanOption(K, T, contract_type="call")
put_option = EuropeanOption(K, T, contract_type="put")

call_price, call_delta, call_gamma, call_theta, x_call = call_option.price_CN(S0=S0, r=r, sigma=sigma, s_steps=s_steps, t_steps=t_steps, version='v6', return_greeks=True)
put_price, put_delta, put_gamma, put_theta, x_put = put_option.price_CN(S0=S0, r=r, sigma=sigma, s_steps=s_steps, t_steps=t_steps, version='v6', return_greeks=True)

assert (x_call == x_put).all(), "x grids for call and put should be the same"
x = x_call
# remove boundaries
x = x[1:-1]
# greeks already have boundaries removed

# get a range to plot over
moneyness = x - np.log(K)
moneyness_min = -0.5
moneyness_max = 0.5
min_idx = np.argmin(np.abs(moneyness - moneyness_min))
max_idx = np.argmin(np.abs(moneyness - moneyness_max))
x = x[min_idx:max_idx+1]
S = np.exp(x)

call_delta = call_delta[min_idx:max_idx+1]
call_gamma = call_gamma[min_idx:max_idx+1]
call_theta = call_theta[min_idx:max_idx+1]

put_delta = put_delta[min_idx:max_idx+1]
put_gamma = put_gamma[min_idx:max_idx+1]
put_theta = put_theta[min_idx:max_idx+1]


In [4]:
bs_call_delta, bs_call_gamma, bs_theta_call = bs_greeks(S, K, r, sigma, T, type="call")
bs_put_delta, bs_put_gamma, bs_theta_put = bs_greeks(S, K, r, sigma, T, type="put")

df = pd.DataFrame({
    'S': S,
    'CN_call_Delta': call_delta,
    'CN_call_Gamma': call_gamma,
    'CN_call_Theta': call_theta,
    'CN_put_Delta': put_delta,
    'CN_put_Gamma': put_gamma,
    'CN_put_Theta': put_theta,

    'BS_call_Delta': bs_call_delta,
    'BS_call_Gamma': bs_call_gamma,
    'BS_call_Theta': bs_theta_call,
    'BS_put_Delta': bs_put_delta,
    'BS_put_Gamma': bs_put_gamma,
    'BS_put_Theta': bs_theta_put
})

def _error(df, greek, option_type="call"):
    return df[f'CN_{option_type}_{greek}'] - df[f'BS_{option_type}_{greek}']


def _relative_error(df, greek, option_type="call"):
    return _error(df, greek, option_type) / df[f'BS_{option_type}_{greek}']



In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _compare(df, greek, option_type="call"):

    fig = make_subplots(
        rows=3,cols=2,
        subplot_titles=(greek, None, f"{greek} error", "Details", f"{greek} percentage error", None),
    )


    # TOP LEFT PLOT FIRST:
    # plot the greek for the call and the put, for both CN and BS

    # top left plot: CN call greek
    fig.add_trace(
        go.Scatter(
            x=df["S"], y=df[f"CN_{option_type}_{greek}"], mode="lines", name=f"CN {option_type} {greek}", line=dict(color='#1f77b4'),
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # top left plot again: dashed red line for BS call greek
    fig.add_trace(
        go.Scatter(
            x=df["S"], y=df[f"BS_{option_type}_{greek}"], mode="lines", name=f"BS {option_type} {greek}", line=dict(color="red", dash="dash"),
            showlegend=True,
        ),
        row=1,
        col=1,
    )


    # TOP RIGHT PLOT
    # plot the error for the call

    # top right plot: call greek error
    fig.add_trace(
        go.Scatter(
            x=df["S"], y=_error(df, greek, option_type), mode="lines", name=f"CN {option_type} {greek} Error", line=dict(color='#1f77b4'), showlegend=False,
        ),
        row=2,
        col=1,
    )


    # BOTTOM LEFT PLOT: WILL UPDATE IT LATER WITH DETAILS


    # BOTTOM RIGHT PLOT

    # bottom right plot: call greek percentage error
    fig.add_trace(
        go.Scatter(
            x=df["S"], y=_relative_error(df, greek, option_type), mode="lines", name=f"CN {option_type} {greek} Percentage Error", line=dict(color='#1f77b4'), showlegend=False,
        ),
        row=3,
        col=1,
    )


    fig.update_layout(
        title={
            "text": f"{greek} for a European {option_type.capitalize()}: Crank-Nicolson vs Black-Scholes Baseline",
            "x": 0.5,
            "xanchor": "center",
        },
        height=750,
    )


    fig.update_yaxes(tickformat=".6f")
    fig.update_yaxes(
        title_text="Percentage Error",
        row=1,
        col=3,
    )



    fig.add_vline(
        x=K,
        line_dash="dash",
        line_color="yellow",
    )

    # Label the strike only once
    fig.add_vline(
        x=K,
        row=1,
        col=1,
        line_dash="dash",
        line_color="yellow",
        name=f"Strike = {K}",
        showlegend=True,
    )

    # bottom left plot: use area to display the parameters used in the numerical analysis
    x0, x1 = fig.layout.xaxis4.domain
    y0, y1 = fig.layout.yaxis4.domain

    fig.add_annotation(
        text=(
            f"s_steps = {s_steps}<br>"
            f"t_steps = {t_steps}<br>"
            f"K = {K}<br>"
            f"σ = {sigma}<br>"
            f"T = {T}<br>"
            f"r = {r}<br>"
        ),

        # centre of row 2, col 1
        x=(x0 + x1) / 2,
        y=(y0 + y1) / 2,

        xref="paper",
        yref="paper",

        showarrow=False,
        align="left",
        xanchor="center",
        yanchor="middle",

        font=dict(size=14),


    )


    fig.show()

In [6]:
_compare(df, "Delta", option_type="call")

In [8]:
GREEKS = ("Delta", "Gamma", "Theta")
COLOURS = {"Delta": "#1f77b4", "Gamma": "#2ca02c", "Theta": "#ff7f0e"}


def _worst_errors(s_steps, t_steps, version, moneyness_min=-0.5, moneyness_max=0.5):
    """
    Worst |error| in each greek over the moneyness window, against the BS baseline.

    Returns (dx, abs_err, rel_err), where the two error terms are dicts keyed by
    greek name. Uses the call, since by put-call parity the put has an identical
    absolute error.
    """
    option = EuropeanOption(K, T, contract_type="call")

    _, delta, gamma, theta, x = option.price_CN(
        S0=S0, r=r, sigma=sigma, s_steps=s_steps, t_steps=t_steps,
        version=version, return_greeks=True,
    )

    # remove boundaries; the greeks already have them removed
    x = x[1:-1]
    S = np.exp(x)

    moneyness = x - np.log(K)
    window = (moneyness >= moneyness_min) & (moneyness <= moneyness_max)

    cn = dict(zip(GREEKS, (delta, gamma, theta)))
    bs = dict(zip(GREEKS, bs_greeks(S, K, r, sigma, T, type="call")))

    abs_err = {g: np.abs(cn[g][window] - bs[g][window]).max() for g in GREEKS}
    rel_err = {g: np.abs((cn[g][window] - bs[g][window]) / bs[g][window]).max() for g in GREEKS}

    return x[1] - x[0], abs_err, rel_err


# LEFT PANEL: spatial convergence.
# hold t_steps large so the dx truncation error dominates, then halve dx repeatedly
S_STEPS_SWEEP = [125, 250, 500, 1000, 2000]
T_STEPS_FIXED = 4000

dxs = []
convergence = {greek: [] for greek in GREEKS}

for n in S_STEPS_SWEEP:
    dx, abs_err, _ = _worst_errors(n, T_STEPS_FIXED, version="v6")
    dxs.append(dx)
    for greek in GREEKS:
        convergence[greek].append(abs_err[greek])

dxs = np.array(dxs)


# RIGHT PANEL: the value of Rannacher smoothing.
# hold s_steps fixed and shrink t_steps, so dt grows relative to dx^2. gamma is the
# worst affected greek, since it is the second derivative of the payoff kink
S_STEPS_FIXED = 500
T_STEPS_SWEEP = [10, 25, 50, 100, 250, 500]

rannacher = {"v3": [], "v6": []}

for version in rannacher:
    for n in T_STEPS_SWEEP:
        _, _, rel_err = _worst_errors(S_STEPS_FIXED, n, version=version)
        rannacher[version].append(rel_err["Gamma"] * 100)


fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Spatial convergence of the greeks",
        "Effect of Rannacher smoothing on gamma",
    ),
)

# left panel: one line per greek, with the fitted order in the legend
for greek in GREEKS:
    order = np.polyfit(np.log(dxs), np.log(convergence[greek]), 1)[0]
    fig.add_trace(
        go.Scatter(
            x=dxs,
            y=convergence[greek],
            mode="lines+markers",
            name=f"{greek} (fitted order {order:.2f})",
            line=dict(color=COLOURS[greek]),
        ),
        row=1,
        col=1,
    )

# left panel again: a slope 2 guide, anchored to the first theta point
fig.add_trace(
    go.Scatter(
        x=dxs,
        y=convergence["Theta"][0] * (dxs / dxs[0]) ** 2,
        mode="lines",
        name="slope 2 reference",
        line=dict(color="grey", dash="dot"),
    ),
    row=1,
    col=1,
)

# right panel: gamma error with and without the implicit Euler startup steps
for version, colour, label in [("v3", "red", "v3 (no Rannacher)"),
                               ("v6", "#1f77b4", "v6 (Rannacher)")]:
    fig.add_trace(
        go.Scatter(
            x=T_STEPS_SWEEP,
            y=rannacher[version],
            mode="lines+markers",
            name=label,
            line=dict(color=colour),
        ),
        row=1,
        col=2,
    )

fig.update_xaxes(type="log", title_text="dx (log grid spacing)", row=1, col=1)
fig.update_yaxes(type="log", title_text="worst absolute error", tickformat=".0e", row=1, col=1)

fig.update_xaxes(type="log", title_text="t_steps", row=1, col=2)
fig.update_yaxes(type="log", title_text="worst gamma error (%)", row=1, col=2)

fig.update_layout(
    title={
        "text": (
            "Crank-Nicolson Greeks: Convergence and the Value of Rannacher Smoothing"
            f"<br><sub>European call, K = {K}, T = {T}, r = {r}, σ = {sigma}"
            f" &nbsp;|&nbsp; left: t_steps = {T_STEPS_FIXED} fixed"
            f" &nbsp;|&nbsp; right: s_steps = {S_STEPS_FIXED} fixed"
            f" &nbsp;|&nbsp; worst error over moneyness [{-0.5}, {0.5}]</sub>"
        ),
        "x": 0.5,
        "xanchor": "center",
    },
    height=550,
)

fig.show()